# DSC PJATK Catalogue Quality Challenge
CPU baseline: EDA, text and numeric features, five seller-disjoint folds.
Attach the competition data and run all cells. Internet and GPU are not required.
ROC-AUC is the official metric; Average Precision provides additional minority-class diagnostics.
Training target: 0 = no controlled corruption; 1 = one controlled corruption. Test target is hidden.
Submission target is a probability in [0,1]. Missing values and unusual prices alone are not labels.
The first-pcs quantity feature is a limited starter heuristic: inspect free accessories and word counts.
Do not tune parameters against the public leaderboard.


In [1]:
"""CPU baseline. Run: python baseline_solution.py --data-dir data/public"""
import argparse
import json
import hashlib
import platform
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42


def seller_folds(sellers, n_splits=5, seed=SEED):
    """Stable group assignment across sklearn versions; no target information used."""
    assignment = np.array([int(hashlib.sha256(f'{seed}:{s}'.encode('utf-8')).hexdigest()[:16], 16)
                           % n_splits for s in sellers.astype(str)])
    for fold in range(n_splits):
        fit, valid = np.flatnonzero(assignment != fold), np.flatnonzero(assignment == fold)
        if not len(fit) or not len(valid):
            raise ValueError('Empty seller fold; use more groups or fewer folds')
        yield fit, valid


def features(df):
    x = df.copy()
    title = x.title.fillna('').astype(str).str.lower()
    desc = x.description.fillna('').astype(str).str.lower()
    x['text'] = title + ' ' + desc
    x['title_length'] = title.str.len()
    x['description_length'] = desc.str.len()
    # Missing observations remain missing; imputation is learned within each fold.
    for col in ['price_pln', 'shipping_weight_kg', 'pack_size']:
        x['log_' + col] = np.log1p(x[col].clip(lower=0))
    x['price_per_item'] = np.log1p(x.price_pln / x.pack_size.clip(lower=1))
    x['mass_per_item'] = np.log1p(x.shipping_weight_kg / x.pack_size.clip(lower=1))
    # Starter hint, not a guaranteed main-item count: free accessories and words
    # require a better parser. Evaluate improvements using held-out sellers.
    extracted = x.text.str.extract(r'(\d+)\s*pcs', expand=False)
    x['text_pack'] = pd.to_numeric(extracted, errors='coerce')
    x['pack_difference'] = (x.text_pack - x.pack_size).abs()
    # Useful text-category interactions; no vocabulary or statistics learned here.
    x['category_text'] = [
        ' '.join(f'{cat}__{token}' for token in re.findall(r'\w+', text))
        for cat, text in zip(x.declared_category.fillna('missing'), x.text)
    ]
    return x.replace({pd.NA: np.nan})


def make_model():
    numeric = ['log_price_pln', 'log_shipping_weight_kg', 'log_pack_size',
               'price_per_item', 'mass_per_item', 'text_pack', 'pack_difference',
               'title_length', 'description_length']
    categorical = ['declared_category', 'brand', 'condition']
    preprocessing = ColumnTransformer([
        ('text', TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=18000,
                                 sublinear_tf=True), 'text'),
        ('interactions', TfidfVectorizer(min_df=3, max_features=12000), 'category_text'),
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                              ('scale', StandardScaler(with_mean=False))]), numeric),
        ('categories', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                                 ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical)
    ], sparse_threshold=1.)
    return Pipeline([('preprocess', preprocessing),
                     ('model', LogisticRegression(C=2., max_iter=600, solver='liblinear',
                                                  random_state=SEED))])


def resolve_data_dir():
    local = Path('data/public')
    if (local / 'train.csv').exists():
        return local
    candidates = sorted(Path('/kaggle/input').rglob('train.csv'))
    candidates = [p.parent for p in candidates if (p.parent / 'sample_submission.csv').exists()]
    if len(candidates) != 1:
        raise ValueError('Specify --data-dir: could not identify one competition dataset')
    return candidates[0]


def run(data_dir, output_dir='baseline_output'):
    started = time.perf_counter()
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(data_dir / 'train.csv')
    test = pd.read_csv(data_dir / 'test.csv')
    sample = pd.read_csv(data_dir / 'sample_submission.csv')
    assert train.target.isin([0, 1]).all()
    assert set(train.seller_id).isdisjoint(test.seller_id)
    assert train.id.is_unique and test.id.is_unique
    assert set(train.id).isdisjoint(test.id)
    assert sample.id.tolist() == test.id.tolist()
    print('Shapes:', train.shape, test.shape)
    print('Target proportions:', train.target.value_counts(normalize=True).to_dict())
    print('Missing fraction:', train.isna().mean().round(3).to_dict())
    print('Categories:', train.declared_category.value_counts(dropna=False).to_dict())
    print('Numeric distribution:\n', train.select_dtypes('number').describe().round(2))
    x, xt, y = features(train.drop(columns='target')), features(test), train.target.to_numpy()
    oof, predictions, scores = np.zeros(len(train)), np.zeros(len(test)), []
    fold_assignment = np.zeros(len(train), dtype=int)
    for fold, (fit, valid) in enumerate(seller_folds(train.seller_id), 1):
        assert set(train.iloc[fit].seller_id).isdisjoint(train.iloc[valid].seller_id)
        fold_assignment[valid] = fold
        model = make_model()
        model.fit(x.iloc[fit], y[fit])
        oof[valid] = model.predict_proba(x.iloc[valid])[:, 1]
        predictions += model.predict_proba(xt)[:, 1] / 5
        score = roc_auc_score(y[valid], oof[valid])
        scores.append(float(score))
        print(f'Fold {fold}: ROC-AUC={score:.6f}')
    report = {'fold_roc_auc': scores, 'mean_roc_auc': float(np.mean(scores)),
              'std_roc_auc': float(np.std(scores)), 'oof_roc_auc': float(roc_auc_score(y, oof)),
              'oof_average_precision': float(average_precision_score(y, oof)),
              'elapsed_seconds': time.perf_counter() - started,
              'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                           'pandas': pd.__version__, 'sklearn': sklearn.__version__},
              'seed': SEED,
              'fold_policy': 'SHA256(seed:seller_id) first 16 hex digits modulo 5',
              'data_sha256': {name: hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
                              for name in ['train.csv', 'test.csv', 'sample_submission.csv']},
              'fold_assignment_sha256': hashlib.sha256(
                  pd.DataFrame({'id': train.id, 'fold': fold_assignment}).to_csv(index=False, lineterminator='\n').encode('utf-8')
              ).hexdigest()}
    sample['target'] = predictions
    assert np.isfinite(predictions).all() and ((predictions >= 0) & (predictions <= 1)).all()
    sample.to_csv(output_dir / 'submission.csv', index=False)
    pd.DataFrame({'id': train.id, 'target': y, 'prediction': oof,
                  'fold': fold_assignment}).to_csv(output_dir / 'oof.csv', index=False)
    (output_dir / 'cv_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report




## Validation and submission
Each fold fits its own imputation, TF-IDF vocabulary and model.
The CV score uses only listings held out from the corresponding model.
The submission averages five models; preserve this procedure when reproducing your result.


In [2]:
data_dir = resolve_data_dir()
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('baseline_output')
report = run(data_dir, output_dir)
print('Submission:', output_dir / 'submission.csv')


Shapes: (16000, 11) (10000, 10)
Target proportions: {0: 0.8299375, 1: 0.1700625}
Missing fraction: {'id': 0.0, 'seller_id': 0.0, 'title': 0.029, 'description': 0.042, 'declared_category': 0.023, 'brand': 0.059, 'condition': 0.0, 'price_pln': 0.039, 'shipping_weight_kg': 0.06, 'pack_size': 0.03, 'target': 0.0}
Categories: {'computers': 2759, 'audio': 2535, 'kitchen': 2253, 'sport': 1837, 'books': 1691, 'beauty': 1622, 'garden': 1331, 'pets': 863, 'music': 735, nan: 374}
Numeric distribution:
        price_pln  shipping_weight_kg  pack_size    target
count   15377.00            15038.00   15515.00  16000.00
mean      145.47                0.98       1.52      0.17
std       212.01                1.65       1.00      0.38
min         0.97                0.02       1.00      0.00
25%        39.67                0.23       1.00      0.00
50%        77.51                0.46       1.00      0.00
75%       166.13                0.99       2.00      0.00
max      3464.27               32.83   